In [1]:
import os
os.chdir("../")
%pwd

'd:\\Deep_Learning_Object_Detection\\MLOPs\\pneumonia-segmentation'

In [2]:
from dataclasses import dataclass

@dataclass(frozen=True)
class LLMParams:
    model:      str
    max_tokens: int

@dataclass(frozen=True)
class ClaudeArchitectureConfig:
    prometheus_url: str
    params:         LLMParams

In [3]:
from core.constants import *
from core.utils import read_yaml, create_directories

class ConfigurationManger:
    def __init__(self, 
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])
    
    def get_claude_architecture_config(self) -> ClaudeArchitectureConfig:
        params = self.params.claude_params
        
        return ClaudeArchitectureConfig(
            prometheus_url = params.prometheus_url,
            params = LLMParams(
                model       = params.model,
                max_tokens  = params.max_tokens,
            )
        )

In [4]:
import json, os, sys, httpx, anthropic
from dataclasses import dataclass
from core.logging import logger
from core.exception import CustomException
from dotenv import load_dotenv

load_dotenv()

class ClaudeArchitecture:
    def __init__(self, config: ClaudeArchitectureConfig):
        self.config = config
        self.client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
        
    def _get_topology_snapshot(self) -> dict:
        q = self._query_prometheus
        return {
            "services": {
                "app": {
                    "cpu":     q('rate(process_cpu_seconds_total{job="app"}[5m]) * 100'),
                    "ram":     q('process_resident_memory_bytes{job="app"} / 1024 / 1024'),
                    "latency": q('http_request_duration_seconds{job="app", quantile="0.95"}'),
                    "drift":   q('inference_drift_score{job="app"}'),
                },
                "lstm": {
                    "cpu": q('rate(process_cpu_seconds_total{job="lstm"}[5m]) * 100'),
                    "ram": q('process_resident_memory_bytes{job="lstm"} / 1024 / 1024'),
                },
                "dqn": {
                    "cpu": q('rate(process_cpu_seconds_total{job="dqn"}[5m]) * 100'),
                    "ram": q('process_resident_memory_bytes{job="dqn"} / 1024 / 1024'),
                }
            }
        }
    
    def _query_prometheus(self, query: str) -> float:
        try: 
            r = httpx.get(
                f"{self.config.prometheus_url}/api/v1/query", 
                params={"query": query}
            )
            
            result = r.json()["data"]["result"]
            return float(result[0]["value"][1]) if result else 0.0
        except Exception as e:
            raise 0.0
    
    
    def get_claude_decision(self) -> dict:
        try:
            topology  = self._get_topology_snapshot()
            
            response  = self.client.messages.create(
                model = self.config.params.model,
                max_tokens = self.config.params.max_tokens,
                messages = [{
                    "role": "user",
                    "content": self._build_prompt(topology)
                }]
            )       
        
            result = self._parse_response(response.content[0].text)
            logger.info(f"Claude architect: {result['action']} — {result['reasoning']}")
            return result
        except Exception as e:
            raise CustomException(e, sys)
        
    def _build_prompt(self, topology: dict) -> str:
        return f"""You are an AI system architect managing a medical image segmentation MLOps platform.
            Current system topology:
            {json.dumps(topology, indent=2)}

            Analyze the system state and decide if architectural evolution is needed.

            Respond ONLY with a JSON object, no explanation, no markdown:
            {{
                "evolution_needed": true/false,
                "action": "none" | "spawn" | "swap" | "rollback",
                "target_service": "app" | "lstm" | "dqn" | "none",
                "parameters": {{}},
                "reasoning": "brief explanation",
                "confidence": 0.0-1.0
            }}

            Rules:
            - spawn: if app latency > 1.0s AND cpu > 70%
            - swap: if drift > 0.8 AND current model is int8
            - rollback: if latency suddenly increased > 2x after recent action
            - none: if system is healthy
            """
    
    def _parse_response(self, raw: str) -> dict:
        cleaned = raw.replace("```json", "").replace("```", "").strip()
        return json.loads(cleaned)

In [5]:
try:
    config = ConfigurationManger()
    claude_config    = config.get_claude_architecture_config()
    claude_architect = ClaudeArchitecture(config=claude_config)
    results = claude_architect.get_claude_decision()
    
    print(results['action'])
    print(results['reasoning'])
except Exception as e:
    raise CustomException(e, sys)

[2026-04-29 10:48:32,245: INFO: __init__: yaml file: config\config.yaml loaded successfully]
[2026-04-29 10:48:32,253: INFO: __init__: yaml file: params.yaml loaded successfully]
[2026-04-29 10:48:32,254: INFO: __init__: created directory at: artifacts]
[2026-04-29 10:48:32,949: INFO: _client: HTTP Request: GET http://localhost:9090/api/v1/query?query=rate%28process_cpu_seconds_total%7Bjob%3D%22app%22%7D%5B5m%5D%29+%2A+100 "HTTP/1.1 200 OK"]
[2026-04-29 10:48:33,215: INFO: _client: HTTP Request: GET http://localhost:9090/api/v1/query?query=process_resident_memory_bytes%7Bjob%3D%22app%22%7D+%2F+1024+%2F+1024 "HTTP/1.1 200 OK"]
[2026-04-29 10:48:33,485: INFO: _client: HTTP Request: GET http://localhost:9090/api/v1/query?query=http_request_duration_seconds%7Bjob%3D%22app%22%2C+quantile%3D%220.95%22%7D "HTTP/1.1 200 OK"]
[2026-04-29 10:48:33,805: INFO: _client: HTTP Request: GET http://localhost:9090/api/v1/query?query=inference_drift_score%7Bjob%3D%22app%22%7D "HTTP/1.1 200 OK"]
[2026-04-